In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table(fitsio.read('/global/cfs/cdirs/desi/spectro/redux/fuji/exposures-fuji.fits', ext='EXPOSURES'))
print(len(cat), len(np.unique(cat['EXPID'])))

2470 2470


In [4]:
subsamp_dict = {'80605': [[74781, 73702],
  [67975, 74782],
  [68292, 74779],
  [68291, 68290],
  [74783, 74780]],
 '80606': [[68630, 67968], [67971, 68812], [67969, 68813, 67970]],
 '80607': [[67768, 68844],
  [68027, 68845],
  [67767, 68028, 67766],
  [68847, 67744],
  [67765, 68846]],
 '80608': [[68025, 68328],
  [67770, 68026],
  [67771, 68327],
  [67769, 68024],
  [68842, 68023],
  [68491, 68841]],
 '80609': [[68489, 67781],
  [67784, 68338],
  [68336, 68490],
  [68064, 68334],
  [68340, 67783],
  [68063, 68065]],
 '80610': [[68333, 68477],
  [68332, 68331],
  [68330, 68042],
  [68488, 68041],
  [68040, 75116]]}

In [5]:
# exp_list = sorted([i for k in subsamp_dict.values() for j in k for i in j])
# print(len(exp_list), len(np.unique(exp_list)))

# mask = np.in1d(cat['EXPID'], exp_list)
# print(np.sum(mask))
# # cat[mask]

In [6]:
# mask_bgs = cat['TILEID']==80613
# mask_bgs &= cat['BGS_EFFTIME_BRIGHT']/cat['EXPTIME'] > 0.2

# mask_dark = cat['TILEID']!=80613
# mask_dark &= cat['EFFTIME_SPEC']/cat['EXPTIME'] > 0.3

# mask = mask_bgs | mask_dark
# cat = cat[mask]
# print(len(cat))

In [7]:
print('                 ELG_EFFTIME_DARK  BGS_EFFTIME_BRIGHT   EFFTIME_DARK_GFA   EFFTIME_BRIGHT_GFA')
print()

# for tileid in subsamp_dict.keys():
for tileid_str in ['80605', '80607', '80609', '80606', '80608', '80610']:
    tileid = int(tileid_str)
    mask = cat['TILEID']==tileid
    print(cat['FAPRGRM'][mask][0].upper(), tileid)
    for subset_index, exp_list in enumerate(subsamp_dict[tileid_str]):
        # print(exp_list)
        mask = np.in1d(cat['EXPID'], exp_list)
        if np.sum(mask)>1:
            print_str = 'exposures'
        else:
            print_str = 'exposure '
        print('subset {}: {} {} {:11.0f} {:19.0f} {:18.0f} {:20.0f}'.format(subset_index+1, np.sum(mask), print_str, np.sum(cat['ELG_EFFTIME_DARK'][mask]), np.sum(cat['BGS_EFFTIME_BRIGHT'][mask]), np.sum(cat['EFFTIME_DARK_GFA'][mask]), np.sum(cat['EFFTIME_BRIGHT_GFA'][mask])))
    print()

                 ELG_EFFTIME_DARK  BGS_EFFTIME_BRIGHT   EFFTIME_DARK_GFA   EFFTIME_BRIGHT_GFA

LRGQSO 80605
subset 1: 2 exposures         941                 938                  0                    0
subset 2: 2 exposures        1031                1007                  0                    0
subset 3: 2 exposures        1098                1130                  0                    0
subset 4: 2 exposures        1119                1195                  0                    0
subset 5: 2 exposures         974                 964                  0                    0

LRGQSO 80607
subset 1: 2 exposures        1067                 977                  0                    0
subset 2: 2 exposures        1065                 979                  0                    0
subset 3: 3 exposures        1063                 999                  0                    0
subset 4: 2 exposures        1085                 998                  0                    0
subset 5: 2 exposures        104

In [8]:
# Print summary
# for tileid_str in subsamp_dict.keys():
for tileid_str in ['80605', '80607', '80609', '80606', '80608', '80610']:
    tileid = int(tileid_str)
    mask = cat['TILEID']==tileid
    tile_flavor = cat['FAPRGRM'][mask][0].upper()
    n_exp = np.sum(mask)
    if tileid==80613:
        nom_depth = 180.
        depth_col = 'BGS_EFFTIME_BRIGHT'
    else:
        nom_depth = 1000.
        depth_col = 'EFFTIME_SPEC'
    tot_depth = np.sum(cat[depth_col][mask])
    print('Tile {} ({}):'.format(tileid, tile_flavor))
    print('all:       n_exp={:2}  {}={:5.0f}s'.format(n_exp, depth_col, tot_depth))
    subsets = subsamp_dict[tileid_str]
    for index, subset in enumerate(subsets):
        mask = np.in1d(cat['EXPID'], subset)
        subset_depth = np.sum(cat[depth_col][mask])
        print('subset {}:  n_exp={:2}  {}={:5.0f}s'.format(index+1, len(subset), depth_col, subset_depth))
    mask = (cat['TILEID']==tileid) & (~np.in1d(cat['EXPID'], np.concatenate(subsets)))
    unused_depth = np.sum(cat[depth_col][mask])
    print('unused:    n_exp={:2}  {}={:5.0f}s'.format(np.sum(mask), depth_col, unused_depth))
    print()

Tile 80605 (LRGQSO):
all:       n_exp=24  EFFTIME_SPEC= 6872s
subset 1:  n_exp= 2  EFFTIME_SPEC=  941s
subset 2:  n_exp= 2  EFFTIME_SPEC= 1031s
subset 3:  n_exp= 2  EFFTIME_SPEC= 1098s
subset 4:  n_exp= 2  EFFTIME_SPEC= 1119s
subset 5:  n_exp= 2  EFFTIME_SPEC=  974s
unused:    n_exp=14  EFFTIME_SPEC= 1709s

Tile 80607 (LRGQSO):
all:       n_exp=15  EFFTIME_SPEC= 9374s
subset 1:  n_exp= 2  EFFTIME_SPEC= 1067s
subset 2:  n_exp= 2  EFFTIME_SPEC= 1065s
subset 3:  n_exp= 3  EFFTIME_SPEC= 1063s
subset 4:  n_exp= 2  EFFTIME_SPEC= 1085s
subset 5:  n_exp= 2  EFFTIME_SPEC= 1048s
unused:    n_exp= 4  EFFTIME_SPEC= 4046s

Tile 80609 (LRGQSO):
all:       n_exp=15  EFFTIME_SPEC= 7819s
subset 1:  n_exp= 2  EFFTIME_SPEC=  954s
subset 2:  n_exp= 2  EFFTIME_SPEC= 1034s
subset 3:  n_exp= 2  EFFTIME_SPEC=  952s
subset 4:  n_exp= 2  EFFTIME_SPEC= 1016s
subset 5:  n_exp= 2  EFFTIME_SPEC= 1032s
subset 6:  n_exp= 2  EFFTIME_SPEC=  926s
unused:    n_exp= 3  EFFTIME_SPEC= 1905s

Tile 80606 (ELG):
all:       n_e